[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/10_Deployment/02_Cloud_Deployment/Cloud_Deployment_Deep_Dive.ipynb)

# Cloud Deployment with ONNX Runtime — Deep Dive

A comprehensive treatment of deploying ONNX models in cloud environments:
throughput optimization, auto-scaling policies, dynamic batching, containerization,
Kubernetes orchestration, and production monitoring at scale.

---

## Table of Contents

| # | Section | Key Topics |
|---|---------|------------|
| 1 | [Cloud Inference Fundamentals](#1) | Throughput vs latency, SLA modeling |
| 2 | [Throughput Optimization Theory](#2) | QPS equations, Little's Law, GPU utilization |
| 3 | [Dynamic Batching Mathematics](#3) | Batch scheduling, padding waste, timeout tradeoffs |
| 4 | [Auto-Scaling Policies](#4) | Reactive vs predictive, scaling equations |
| 5 | [Containerization Architecture](#5) | Docker, multi-stage builds, ORT configuration |
| 6 | [Kubernetes Orchestration](#6) | HPA, resource quotas, GPU scheduling |
| 7 | [Load Balancing and Routing](#7) | Request distribution, session affinity |
| 8 | [Monitoring and Observability](#8) | SLI/SLO framework, latency histograms |
| 9 | [Cost Optimization](#9) | Spot instances, right-sizing, precision tradeoffs |
| 10 | [Production Deployment Patterns](#10) | Blue-green, canary, shadow |

---

![ONNX Deployment Targets](assets/onnx_deployment_targets.png)

<a id='1'></a>
## 1. Cloud Inference Fundamentals

Cloud deployment optimizes for **throughput** (requests served per unit time) while maintaining **latency SLAs** (response time guarantees). Unlike edge deployment where the constraint is hardware, cloud deployment's constraint is **cost-efficiency at scale**.

### The Cloud Serving Triangle

```
                  Throughput
                     /\
                    /  \
                   /    \
                  / PICK \
                 /  TWO   \
                /          \
               /____________\
         Latency           Cost
```

### Service Level Objectives (SLOs)

Cloud ML services are governed by SLOs that define acceptable performance:

$$\text{Availability} = \frac{T_{\text{total}} - T_{\text{downtime}}}{T_{\text{total}}} \geq 99.9\%$$

$$P(T_{\text{response}} \leq T_{\text{SLA}}) \geq 0.99 \quad \text{(p99 latency guarantee)}$$

### Latency Decomposition in Cloud

$$T_{\text{e2e}} = T_{\text{network}} + T_{\text{queue}} + T_{\text{preprocess}} + T_{\text{inference}} + T_{\text{postprocess}} + T_{\text{serialize}}$$

In cloud deployments, $T_{\text{queue}}$ often dominates under load — this is where batching and scaling interact.

### Cloud vs Edge: Fundamental Differences

| Dimension | Edge | Cloud |
|-----------|------|-------|
| **Batch size** | 1 (streaming) | 1–256 (dynamic) |
| **Scaling** | Fixed hardware | Elastic (0 to N replicas) |
| **GPU** | Embedded (Jetson) | Data center (A100/H100) |
| **Optimization goal** | Fit in memory | Maximize QPS/$ |
| **Failure mode** | Device failure | Cascading overload |
| **Model updates** | OTA (hours) | Rolling deploy (seconds) |

<a id='2'></a>
## 2. Throughput Optimization Theory

### Queries Per Second (QPS)

The fundamental throughput equation for a model serving system:

$$\text{QPS} = \frac{B \cdot N_{\text{replicas}}}{T_{\text{p99}}}$$

where:
- $B$ = effective batch size per inference call
- $N_{\text{replicas}}$ = number of serving replicas
- $T_{\text{p99}}$ = 99th percentile inference latency (seconds)

### Little's Law for ML Serving

Little's Law relates throughput, latency, and concurrency:

$$L = \lambda \cdot W$$

where:
- $L$ = average number of requests in the system (concurrency)
- $\lambda$ = arrival rate (QPS)
- $W$ = average time a request spends in the system

For a serving system with $N$ workers:

$$\lambda_{\max} = \frac{N}{W} = \frac{N_{\text{replicas}} \cdot C_{\text{threads}}}{T_{\text{avg}}}$$

### GPU Utilization and Throughput

GPU utilization $U$ determines cost-efficiency:

$$U = \frac{\text{Actual FLOPS}}{\text{Peak FLOPS}} = \frac{\text{FLOPs}(\text{model}) \cdot \lambda}{\pi_{\text{GPU}}}$$

For an A100 (312 TFLOPS FP16) serving a model requiring 20 GFLOPS per inference:

$$U = \frac{20 \times 10^9 \cdot \lambda}{312 \times 10^{12}} \implies \lambda_{\max} = \frac{312 \times 10^{12}}{20 \times 10^9} = 15{,}600 \text{ QPS}$$

But memory bandwidth limits this in practice. For memory-bound models:

$$\lambda_{\text{practical}} = \frac{\text{Memory Bandwidth}}{\text{Bytes per inference}} = \frac{2 \text{ TB/s}}{M_{\text{model}} + M_{\text{activations}}}$$

### Amdahl's Law for Batched Inference

Not all operations benefit equally from batching. If fraction $f$ of compute is parallelizable:

$$\text{Speedup}(B) = \frac{1}{(1-f) + \frac{f}{B}}$$

For transformer models, attention is $O(B \cdot T^2 \cdot d)$ — highly parallelizable — but tokenization and postprocessing are serial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Throughput modeling for cloud inference
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: QPS vs replicas at different batch sizes
replicas = np.arange(1, 21)
t_p99_ms = 50  # 50ms p99 latency
batch_sizes = [1, 4, 8, 16, 32]

for B in batch_sizes:
    # Latency increases sub-linearly with batch (GPU parallel)
    t_batch = t_p99_ms * (1 + 0.3 * np.log2(B))  # ms
    qps = B * replicas / (t_batch / 1000)
    axes[0].plot(replicas, qps, 'o-', markersize=4, label=f'B={B}')

axes[0].set_xlabel('Number of Replicas')
axes[0].set_ylabel('Throughput (QPS)')
axes[0].set_title('QPS = B × N_replicas / T_p99')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Middle: GPU utilization vs batch size
batch_range = np.arange(1, 65)
peak_flops = 312e12  # A100 FP16
model_flops = 20e9  # 20 GFLOP model

# Utilization increases with batch due to better parallelism
utilization = 1 - np.exp(-batch_range / 16)  # Saturates around B=32-64
effective_qps = utilization * peak_flops / model_flops

axes[1].plot(batch_range, utilization * 100, 'b-', linewidth=2)
axes[1].axhline(y=80, color='green', linestyle='--', alpha=0.7, label='Target: 80%')
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('GPU Utilization (%)')
axes[1].set_title('GPU Utilization vs Batch Size (A100)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Right: Amdahl's Law for batched inference
f_values = [0.7, 0.8, 0.9, 0.95, 0.99]
B_range = np.arange(1, 129)

for f in f_values:
    speedup = 1 / ((1 - f) + f / B_range)
    axes[2].plot(B_range, speedup, linewidth=1.5, label=f'f={f}')

axes[2].set_xlabel('Batch Size')
axes[2].set_ylabel('Speedup')
axes[2].set_title("Amdahl's Law: Batching Speedup")
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log', base=2)

plt.tight_layout()
plt.savefig('cloud_throughput_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"At B=16, 10 replicas, T_p99=50ms: QPS = {16 * 10 / 0.05:.0f}")
print(f"At B=32, 10 replicas, T_p99=65ms: QPS = {32 * 10 / 0.065:.0f}")

<a id='3'></a>
## 3. Dynamic Batching Mathematics

Dynamic batching groups incoming requests into batches for GPU-efficient execution. The scheduler must balance **throughput** (larger batches) against **latency** (waiting time).

### Batching Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                    Dynamic Batching Server                        │
│                                                                   │
│  Request    ┌──────────────┐    ┌──────────────┐    ┌────────┐  │
│  Stream ──▶ │ Batch Queue  │──▶ │  Batch       │──▶ │  ORT   │  │
│             │              │    │  Scheduler   │    │ Session│  │
│  req_1 ──▶ │ [r1,r2,r3..] │    │              │    │        │  │
│  req_2 ──▶ │              │    │ Fire when:   │    │ GPU    │  │
│  req_3 ──▶ │              │    │ B=B_max OR   │    │ Infer  │  │
│   ...  ──▶ │              │    │ t>t_timeout  │    │        │  │
│             └──────────────┘    └──────────────┘    └────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### Batch Formation Policy

The scheduler fires a batch when either condition is met:

$$\text{fire}(t) = \mathbb{1}\left[|Q(t)| \geq B_{\max}\right] \lor \mathbb{1}\left[t - t_{\text{first}} \geq \tau_{\text{timeout}}\right]$$

where:
- $|Q(t)|$ = current queue depth
- $B_{\max}$ = maximum batch size
- $t_{\text{first}}$ = arrival time of the oldest request in queue
- $\tau_{\text{timeout}}$ = maximum wait time

### Optimal Timeout Selection

Given arrival rate $\lambda$ (requests/sec), the expected batch size with timeout $\tau$ is:

$$\mathbb{E}[B] = \min\left(B_{\max}, \; 1 + \lambda \cdot \tau\right)$$

The expected waiting time for a request:

$$\mathbb{E}[T_{\text{wait}}] = \frac{\tau}{2} \cdot P(|Q| < B_{\max}) + 0 \cdot P(|Q| \geq B_{\max})$$

### Padding Waste in Variable-Length Batches (NLP)

For NLP models with variable sequence lengths, padding introduces wasted computation:

$$\text{Waste}(\mathbf{L}) = \frac{\sum_{i=1}^{B}(L_{\max} - L_i)}{B \cdot L_{\max}} = 1 - \frac{\bar{L}}{L_{\max}}$$

where $\mathbf{L} = [L_1, ..., L_B]$ are the sequence lengths in the batch.

**Bucketing strategy:** Group sequences by length into buckets to minimize waste:

$$\text{Waste}_{\text{bucketed}} = \frac{1}{K}\sum_{k=1}^{K} \left(1 - \frac{\bar{L}_k}{L_{\max,k}}\right)$$

### Throughput-Latency Tradeoff

The system operates on a Pareto frontier:

$$T_{\text{total}} = \underbrace{T_{\text{wait}}(\tau, \lambda)}_{\text{increases with } \tau} + \underbrace{T_{\text{infer}}(B)}_{\text{sub-linear in } B}$$

Throughput:

$$\text{QPS} = \frac{\mathbb{E}[B]}{T_{\text{infer}}(\mathbb{E}[B])}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dynamic batching simulation
np.random.seed(42)

def simulate_dynamic_batching(arrival_rate, timeout_ms, max_batch, duration_s=10):
    """Simulate dynamic batching and measure metrics."""
    # Generate Poisson arrivals
    n_arrivals = np.random.poisson(arrival_rate * duration_s)
    arrivals = np.sort(np.random.uniform(0, duration_s, n_arrivals))
    
    batches = []
    wait_times = []
    queue = []
    
    i = 0
    t = 0
    timeout_s = timeout_ms / 1000
    
    while i < len(arrivals) or queue:
        # Add all arrivals up to current time
        while i < len(arrivals) and arrivals[i] <= t:
            queue.append(arrivals[i])
            i += 1
        
        # Check fire conditions
        if queue:
            oldest = queue[0]
            if len(queue) >= max_batch or (t - oldest) >= timeout_s:
                batch_size = min(len(queue), max_batch)
                batch_requests = queue[:batch_size]
                queue = queue[batch_size:]
                batches.append(batch_size)
                wait_times.extend([t - req_t for req_t in batch_requests])
        
        t += 0.001  # 1ms time step
        if t > duration_s + 1:
            break
    
    return {
        'avg_batch_size': np.mean(batches) if batches else 0,
        'avg_wait_ms': np.mean(wait_times) * 1000 if wait_times else 0,
        'p99_wait_ms': np.percentile(wait_times, 99) * 1000 if wait_times else 0,
        'n_batches': len(batches),
        'throughput': sum(batches) / duration_s if batches else 0,
    }

# Sweep timeout values
arrival_rate = 100  # 100 requests/sec
max_batch = 32
timeouts = np.linspace(1, 100, 30)  # ms

results = [simulate_dynamic_batching(arrival_rate, t, max_batch) for t in timeouts]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Average batch size vs timeout
axes[0].plot(timeouts, [r['avg_batch_size'] for r in results], 'b-o', markersize=4)
axes[0].axhline(y=max_batch, color='red', linestyle='--', label=f'B_max={max_batch}')
axes[0].set_xlabel('Timeout (ms)')
axes[0].set_ylabel('Average Batch Size')
axes[0].set_title('Batch Size vs Timeout\n(λ=100 req/s)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Wait time vs timeout
axes[1].plot(timeouts, [r['avg_wait_ms'] for r in results], 'g-o', markersize=4, label='Mean')
axes[1].plot(timeouts, [r['p99_wait_ms'] for r in results], 'r-s', markersize=4, label='p99')
axes[1].set_xlabel('Timeout (ms)')
axes[1].set_ylabel('Wait Time (ms)')
axes[1].set_title('Request Wait Time vs Timeout')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Throughput-latency Pareto frontier
# Model: inference time increases with batch size (sub-linear on GPU)
infer_times = [20 * (1 + 0.3 * np.log2(max(r['avg_batch_size'], 1))) for r in results]
total_latencies = [r['avg_wait_ms'] + it for r, it in zip(results, infer_times)]
throughputs = [r['throughput'] for r in results]

sc = axes[2].scatter(total_latencies, throughputs, c=timeouts, cmap='viridis', s=50)
plt.colorbar(sc, ax=axes[2], label='Timeout (ms)')
axes[2].set_xlabel('Average Total Latency (ms)')
axes[2].set_ylabel('Throughput (req/s)')
axes[2].set_title('Throughput-Latency Pareto Front')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dynamic_batching_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"At timeout=10ms: avg batch={results[3]['avg_batch_size']:.1f}, wait={results[3]['avg_wait_ms']:.1f}ms")
print(f"At timeout=50ms: avg batch={results[15]['avg_batch_size']:.1f}, wait={results[15]['avg_wait_ms']:.1f}ms")

<a id='4'></a>
## 4. Auto-Scaling Policies

Auto-scaling adjusts replica count to match traffic demand. The goal is to maintain SLOs while minimizing cost.

### Reactive Scaling (Threshold-Based)

The simplest policy scales based on a utilization metric:

$$N_{\text{desired}} = \left\lceil N_{\text{current}} \cdot \frac{U_{\text{current}}}{U_{\text{target}}} \right\rceil$$

For example, with target CPU utilization $U_{\text{target}} = 0.7$:
- If current utilization is 90% with 5 replicas: $N = \lceil 5 \times 0.9/0.7 \rceil = 7$
- If current utilization is 40% with 10 replicas: $N = \lceil 10 \times 0.4/0.7 \rceil = 6$

### Scale-Up and Scale-Down with Hysteresis

To prevent oscillation (thrashing), scaling policies use cooldown periods and asymmetric thresholds:

$$\text{scale\_up if } U > U_{\text{high}} \text{ for } t > T_{\text{cooldown\_up}}$$
$$\text{scale\_down if } U < U_{\text{low}} \text{ for } t > T_{\text{cooldown\_down}}$$

Typically: $T_{\text{cooldown\_down}} \gg T_{\text{cooldown\_up}}$ (scale up fast, scale down slowly).

### Predictive Scaling

For periodic traffic patterns (e.g., daily peaks), predictive scaling pre-provisions:

$$N(t) = \max\left(N_{\text{min}}, \; \left\lceil \frac{\hat{\lambda}(t + \Delta t_{\text{lead}})}{\text{QPS}_{\text{per\_replica}}} \right\rceil\right)$$

where $\hat{\lambda}(t)$ is the predicted arrival rate and $\Delta t_{\text{lead}}$ accounts for instance boot time.

### GPU-Aware Scaling Signals

CPU utilization is a poor proxy for GPU-bound inference. Better signals:

| Signal | Formula | Why |
|--------|---------|-----|
| Queue depth | $|Q| > B_{\max} \cdot k$ | Direct demand indicator |
| Inference latency | $T_{\text{p95}} > 0.8 \cdot T_{\text{SLA}}$ | SLO proximity |
| GPU utilization | $U_{\text{GPU}} > 0.85$ | Hardware saturation |
| Batch fill rate | $\mathbb{E}[B] / B_{\max} > 0.9$ | Demand exceeds capacity |

### Scaling Formula with Cold Start Compensation

New replicas take time to become ready (model loading). During this window, existing replicas bear extra load:

$$N_{\text{provision}} = N_{\text{desired}} + \left\lceil \frac{\lambda \cdot T_{\text{cold\_start}}}{\text{QPS}_{\text{per\_replica}}} \right\rceil$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Auto-scaling simulation
np.random.seed(42)

# Simulate 24-hour traffic pattern
hours = np.linspace(0, 24, 1440)  # minute resolution

# Realistic daily traffic: peak at 10am and 2pm, low at night
base_traffic = 50
traffic = (base_traffic + 
           80 * np.exp(-0.5 * ((hours - 10) / 2)**2) +  # Morning peak
           60 * np.exp(-0.5 * ((hours - 14) / 1.5)**2) +  # Afternoon peak
           np.random.randn(len(hours)) * 5)  # Noise
traffic = np.maximum(traffic, 10)

# Scaling parameters
qps_per_replica = 30
u_target = 0.7
min_replicas = 2
max_replicas = 20
cooldown_up = 2  # minutes
cooldown_down = 10  # minutes

# Simulate reactive scaling
replicas_reactive = np.zeros(len(hours))
replicas_reactive[0] = min_replicas
last_scale_time = -100

for i in range(1, len(hours)):
    current_replicas = replicas_reactive[i-1]
    utilization = traffic[i] / (current_replicas * qps_per_replica)
    desired = int(np.ceil(current_replicas * utilization / u_target))
    desired = np.clip(desired, min_replicas, max_replicas)
    
    minutes_since_scale = i - last_scale_time
    if desired > current_replicas and minutes_since_scale >= cooldown_up:
        replicas_reactive[i] = desired
        last_scale_time = i
    elif desired < current_replicas and minutes_since_scale >= cooldown_down:
        replicas_reactive[i] = desired
        last_scale_time = i
    else:
        replicas_reactive[i] = current_replicas

# Predictive scaling (perfect foresight + lead time)
lead_time = 5  # minutes
replicas_predictive = np.maximum(
    min_replicas,
    np.ceil(np.roll(traffic, -lead_time) / qps_per_replica / u_target)
)
replicas_predictive = np.clip(replicas_predictive, min_replicas, max_replicas)

# Calculate actual utilization under each policy
util_reactive = traffic / (replicas_reactive * qps_per_replica)
util_predictive = traffic / (replicas_predictive * qps_per_replica)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Traffic
axes[0].plot(hours, traffic, 'b-', linewidth=1, alpha=0.8)
axes[0].fill_between(hours, 0, traffic, alpha=0.2, color='blue')
axes[0].set_ylabel('Traffic (QPS)')
axes[0].set_title('24-Hour Traffic Pattern and Auto-Scaling Response')
axes[0].grid(True, alpha=0.3)

# Replica count
axes[1].step(hours, replicas_reactive, 'r-', linewidth=2, label='Reactive', where='post')
axes[1].step(hours, replicas_predictive, 'g-', linewidth=2, label='Predictive', where='post')
ideal_replicas = np.ceil(traffic / qps_per_replica / u_target)
axes[1].plot(hours, ideal_replicas, 'k--', linewidth=1, alpha=0.5, label='Ideal')
axes[1].set_ylabel('Replicas')
axes[1].set_title('Replica Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Utilization
axes[2].plot(hours, util_reactive * 100, 'r-', linewidth=1, alpha=0.7, label='Reactive')
axes[2].plot(hours, util_predictive * 100, 'g-', linewidth=1, alpha=0.7, label='Predictive')
axes[2].axhline(y=u_target * 100, color='black', linestyle='--', label=f'Target ({u_target*100:.0f}%)')
axes[2].axhline(y=100, color='red', linestyle=':', alpha=0.5, label='Overload')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Utilization (%)')
axes[2].set_title('Per-Replica Utilization')
axes[2].set_ylim(0, 150)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('autoscaling_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

# Cost analysis
cost_per_replica_hour = 3.0  # $/hr for GPU instance
reactive_cost = np.sum(replicas_reactive) / 60 * cost_per_replica_hour
predictive_cost = np.sum(replicas_predictive) / 60 * cost_per_replica_hour
print(f"Reactive scaling: ${reactive_cost:.0f}/day")
print(f"Predictive scaling: ${predictive_cost:.0f}/day")
print(f"Overload events (reactive): {np.sum(util_reactive > 1)} minutes")
print(f"Overload events (predictive): {np.sum(util_predictive > 1)} minutes")

<a id='5'></a>
## 5. Containerization Architecture

### Multi-Stage Docker Build for ORT

```
┌─────────────────────────────────────────────────────────┐
│  Stage 1: Builder                                        │
│  ┌─────────────────────────────────────────────────────┐ │
│  │ FROM python:3.11 AS builder                         │ │
│  │ • Install build dependencies                        │ │
│  │ • pip install --target=/install onnxruntime-gpu     │ │
│  │ • Compile custom ops if needed                      │ │
│  └─────────────────────────────────────────────────────┘ │
├─────────────────────────────────────────────────────────┤
│  Stage 2: Runtime                                        │
│  ┌─────────────────────────────────────────────────────┐ │
│  │ FROM nvidia/cuda:12.2-runtime-ubuntu22.04           │ │
│  │ • COPY --from=builder /install /usr/local/lib       │ │
│  │ • COPY model.onnx /models/                          │ │
│  │ • Non-root user, health endpoint                    │ │
│  │ • ENTRYPOINT ["python", "serve.py"]                 │ │
│  └─────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────┘
```

### Image Size Optimization

| Strategy | Typical Savings | Impact |
|----------|----------------|--------|
| Multi-stage build | 40-60% | No build tools in runtime |
| Slim base image | 200-500 MB | Fewer OS packages |
| Selective ORT packages | 100-300 MB | Only needed EPs |
| Model external mount | 100+ MB | Decouple model from image |
| .dockerignore | Variable | No source/tests in context |

### ORT Session Configuration for Cloud

```python
# Production session configuration
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session_options.execution_mode = ort.ExecutionMode.ORT_PARALLEL
session_options.inter_op_num_threads = 4
session_options.intra_op_num_threads = 8
session_options.enable_mem_pattern = True
session_options.enable_cpu_mem_arena = True
```

### Memory Arena Configuration

ORT's memory arena pre-allocates buffers to avoid malloc overhead:

$$M_{\text{arena}} = M_{\text{initial}} + N_{\text{extensions}} \cdot M_{\text{extension}}$$

For GPU serving, pre-allocate based on maximum batch size:

$$M_{\text{arena}} \geq M_{\text{params}} + M_{\text{activation}}(B_{\max}) + M_{\text{workspace}}$$

In [ ]:
import numpy as np
import onnxruntime as ort

# Demonstrate cloud-optimized session configuration
print("=" * 60)
print("ONNX Runtime Cloud Configuration")
print("=" * 60)

# Session options for cloud serving
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.execution_mode = ort.ExecutionMode.ORT_PARALLEL
so.inter_op_num_threads = 4
so.intra_op_num_threads = 8
so.enable_mem_pattern = True
so.enable_cpu_mem_arena = True

print(f"\nGraph optimization: ORT_ENABLE_ALL")
print(f"Execution mode: ORT_PARALLEL")
print(f"Inter-op threads: {so.inter_op_num_threads}")
print(f"Intra-op threads: {so.intra_op_num_threads}")
print(f"Memory pattern: {so.enable_mem_pattern}")
print(f"CPU mem arena: {so.enable_cpu_mem_arena}")

# Available providers
providers = ort.get_available_providers()
print(f"\nAvailable Execution Providers: {providers}")

# Provider selection strategy for cloud
cloud_provider_priority = []
if 'TensorrtExecutionProvider' in providers:
    cloud_provider_priority.append('TensorrtExecutionProvider')
if 'CUDAExecutionProvider' in providers:
    cloud_provider_priority.append('CUDAExecutionProvider')
cloud_provider_priority.append('CPUExecutionProvider')

print(f"Cloud provider priority: {cloud_provider_priority}")

# Thread configuration recommendations
import os
cpu_count = os.cpu_count()
print(f"\nSystem CPU count: {cpu_count}")
print(f"Recommended inter_op_threads: {min(4, cpu_count // 2)}")
print(f"Recommended intra_op_threads: {min(8, cpu_count)}")
print(f"\nNote: For GPU serving, CPU threads handle pre/post-processing.")
print(f"Set intra_op_threads = physical cores for CPU-bound workloads.")

<a id='6'></a>
## 6. Kubernetes Orchestration

### Kubernetes ML Serving Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                        Kubernetes Cluster                             │
│                                                                       │
│  ┌──────────────┐    ┌──────────────────────────────────────────┐   │
│  │   Ingress    │    │          Model Serving Namespace          │   │
│  │  Controller  │    │                                            │   │
│  │              │    │  ┌────────┐ ┌────────┐ ┌────────┐        │   │
│  │  /v1/predict─┼───▶│  │ Pod-1  │ │ Pod-2  │ │ Pod-3  │        │   │
│  │              │    │  │┌──────┐│ │┌──────┐│ │┌──────┐│        │   │
│  │  /health ────┼───▶│  ││ ORT  ││ ││ ORT  ││ ││ ORT  ││        │   │
│  │              │    │  │└──────┘│ │└──────┘│ │└──────┘│        │   │
│  └──────────────┘    │  │  GPU   │ │  GPU   │ │  GPU   │        │   │
│                      │  └────────┘ └────────┘ └────────┘        │   │
│                      │         ▲                                  │   │
│                      │         │ HPA (Horizontal Pod Autoscaler)  │   │
│                      │         │ Scale on: GPU util, queue depth  │   │
│                      └──────────────────────────────────────────┘   │
│                                                                       │
│  ┌──────────────┐    ┌──────────────┐    ┌───────────────────┐      │
│  │  Prometheus  │◀───│   Service    │    │  Model Registry   │      │
│  │  + Grafana   │    │   Monitor    │    │  (S3/GCS bucket)  │      │
│  └──────────────┘    └──────────────┘    └───────────────────┘      │
└─────────────────────────────────────────────────────────────────────┘
```

### Resource Configuration

```yaml
resources:
  requests:
    memory: "4Gi"
    cpu: "2"
    nvidia.com/gpu: "1"
  limits:
    memory: "8Gi"
    cpu: "4"
    nvidia.com/gpu: "1"
```

### Readiness vs Liveness Probes

| Probe | Purpose | Configuration |
|-------|---------|---------------|
| **Liveness** | Restart stuck pods | `GET /health`, period=30s, timeout=5s |
| **Readiness** | Remove from LB during model load | `GET /ready`, initialDelay=60s |
| **Startup** | Long model load tolerance | `GET /ready`, failureThreshold=30 |

Critical: Set `initialDelaySeconds` for readiness to exceed model loading time:

$$T_{\text{initialDelay}} > T_{\text{model\_load}} + T_{\text{warmup}} + T_{\text{margin}}$$

### HPA Configuration for ML

The Horizontal Pod Autoscaler formula:

$$N_{\text{replicas}} = \left\lceil N_{\text{current}} \times \frac{\text{metric}_{\text{current}}}{\text{metric}_{\text{target}}} \right\rceil$$

For GPU-based serving, use custom metrics from DCGM (Data Center GPU Manager):

```yaml
metrics:
- type: Pods
  pods:
    metric:
      name: dcgm_gpu_utilization
    target:
      type: AverageValue
      averageValue: "80"
```

<a id='7'></a>
## 7. Load Balancing and Routing

### Load Balancing Strategies for ML

| Strategy | Formula | Best For |
|----------|---------|----------|
| **Round Robin** | $\text{next} = (i + 1) \mod N$ | Homogeneous workloads |
| **Least Connections** | $\text{next} = \arg\min_i C_i$ | Variable inference times |
| **Weighted** | $P(i) = w_i / \sum w_j$ | Heterogeneous hardware |
| **Latency-aware** | $\text{next} = \arg\min_i \hat{T}_i$ | Mixed model versions |

### Request Routing for A/B Testing

```
                    ┌────────────────┐
   Requests ──────▶│  Traffic Split  │
                    │                │
                    │  90% ─────────▶ Model v2 (stable)
                    │  10% ─────────▶ Model v3 (canary)
                    └────────────────┘
```

### Consistent Hashing for Model Sharding

For ensemble models or large models split across replicas:

$$\text{shard}(\text{request}) = \text{hash}(\text{request\_key}) \mod N_{\text{shards}}$$

With virtual nodes for better distribution:

$$\text{shard}(r) = \min_{v \in V} \{ v : \text{hash}(v) \geq \text{hash}(r) \}$$

### gRPC vs REST for ML Serving

| Dimension | REST (JSON) | gRPC (Protobuf) |
|-----------|-------------|------------------|
| Serialization overhead | $O(n)$ string parsing | $O(1)$ binary decode |
| Payload size (float32 array, 1M elements) | ~8 MB (JSON) | ~4 MB (binary) |
| Streaming | Polling/WebSocket | Native bidirectional |
| Browser support | Universal | grpc-web proxy needed |

For large tensor payloads:

$$\text{Overhead}_{\text{JSON}} = \frac{S_{\text{JSON}} - S_{\text{raw}}}{S_{\text{raw}}} \approx \frac{8N - 4N}{4N} = 100\%$$

gRPC with raw bytes reduces this to near-zero overhead.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load balancing simulation
np.random.seed(42)

n_replicas = 5
n_requests = 1000

# Simulate heterogeneous inference times (some replicas are slower)
replica_speeds = np.array([1.0, 1.0, 0.8, 1.2, 0.6])  # relative speed

def simulate_lb(strategy, n_requests, replica_speeds):
    """Simulate load balancing and return per-replica load."""
    n = len(replica_speeds)
    connections = np.zeros(n)
    latencies = []
    
    for req in range(n_requests):
        if strategy == 'round_robin':
            chosen = req % n
        elif strategy == 'least_connections':
            chosen = np.argmin(connections)
        elif strategy == 'random':
            chosen = np.random.randint(n)
        elif strategy == 'weighted':
            weights = replica_speeds / replica_speeds.sum()
            chosen = np.random.choice(n, p=weights)
        
        # Inference time depends on replica speed and current load
        base_time = 20 / replica_speeds[chosen]  # ms
        load_penalty = connections[chosen] * 5  # ms per concurrent request
        latency = base_time + load_penalty + np.random.exponential(3)
        latencies.append(latency)
        connections[chosen] += 1
        
        # Randomly complete some requests
        if req % 5 == 0:
            completed = np.random.choice(n, size=min(3, n), replace=False)
            connections[completed] = np.maximum(connections[completed] - 1, 0)
    
    return np.array(latencies), connections

strategies = ['round_robin', 'least_connections', 'random', 'weighted']
strategy_labels = ['Round Robin', 'Least Connections', 'Random', 'Weighted']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Latency distributions
for strategy, label in zip(strategies, strategy_labels):
    lats, _ = simulate_lb(strategy, n_requests, replica_speeds)
    axes[0].hist(lats, bins=40, alpha=0.4, label=f'{label} (p99={np.percentile(lats, 99):.0f}ms)')

axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Latency Distribution by LB Strategy\n(Heterogeneous replicas)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# P50/P95/P99 comparison
percentiles = {'p50': [], 'p95': [], 'p99': []}
for strategy in strategies:
    lats, _ = simulate_lb(strategy, n_requests, replica_speeds)
    percentiles['p50'].append(np.percentile(lats, 50))
    percentiles['p95'].append(np.percentile(lats, 95))
    percentiles['p99'].append(np.percentile(lats, 99))

x = np.arange(len(strategies))
width = 0.25
axes[1].bar(x - width, percentiles['p50'], width, label='p50', color='#2ecc71')
axes[1].bar(x, percentiles['p95'], width, label='p95', color='#f39c12')
axes[1].bar(x + width, percentiles['p99'], width, label='p99', color='#e74c3c')
axes[1].set_xticks(x)
axes[1].set_xticklabels(strategy_labels, rotation=15)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency Percentiles by Strategy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('load_balancing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='8'></a>
## 8. Monitoring and Observability

### SLI/SLO Framework for ML Services

**Service Level Indicators (SLIs)** — the metrics you measure:

| SLI | Definition | Target SLO |
|-----|-----------|------------|
| Availability | $\frac{\text{successful requests}}{\text{total requests}}$ | ≥ 99.9% |
| Latency | $P(T \leq T_{\text{threshold}})$ | p99 < 100ms |
| Throughput | Sustained QPS | ≥ $\lambda_{\text{peak}}$ |
| Correctness | $P(\text{output matches reference})$ | ≥ 99.99% |

### Error Budget

$$\text{Error Budget} = 1 - \text{SLO}$$

For 99.9% availability over 30 days:

$$\text{Allowed downtime} = 30 \times 24 \times 60 \times (1 - 0.999) = 43.2 \text{ minutes/month}$$

### Latency Histogram (RED Method)

The RED method tracks three signals:
- **R**ate: requests per second
- **E**rrors: failed requests per second
- **D**uration: latency distribution

### Key Metrics for ORT Serving

```
┌─────────────────────────────────────────────────────────────┐
│                    Monitoring Dashboard                       │
├──────────────────┬──────────────────┬───────────────────────┤
│  REQUEST RATE    │  ERROR RATE      │  LATENCY (p50/p99)    │
│  ╭──────╮       │  ╭──────╮        │  ╭──────╮            │
│  │ ∧∧∧∧ │       │  │  __  │        │  │ ──── │ p50=20ms  │
│  │∧    ∧│       │  │ │  │ │        │  │ ···· │ p99=85ms  │
│  ╰──────╯       │  ╰──────╯        │  ╰──────╯            │
│  150 QPS        │  0.1%            │                       │
├──────────────────┼──────────────────┼───────────────────────┤
│  GPU UTIL       │  BATCH SIZE      │  QUEUE DEPTH          │
│  ╭──────╮       │  ╭──────╮        │  ╭──────╮            │
│  │██████│ 78%   │  │ ∧∧∧∧ │ avg=12│  │  _   │ avg=3     │
│  │██████│       │  │      │        │  │ │ │  │            │
│  ╰──────╯       │  ╰──────╯        │  ╰──────╯            │
└──────────────────┴──────────────────┴───────────────────────┘
```

### Anomaly Detection for Model Drift

Monitor prediction distribution stability using KL divergence:

$$D_{KL}(P_{\text{current}} \| P_{\text{baseline}}) = \sum_{c=1}^{C} P_c^{\text{curr}} \log \frac{P_c^{\text{curr}}}{P_c^{\text{base}}}$$

Alert when:

$$D_{KL} > \mu_{D_{KL}} + 3\sigma_{D_{KL}} \quad \text{(3-sigma rule)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Monitoring simulation: latency histograms and SLO tracking
np.random.seed(42)

# Simulate latency distribution (log-normal is typical for inference)
n_requests = 10000
mu, sigma = np.log(25), 0.5  # Median ~25ms
latencies = np.random.lognormal(mu, sigma, n_requests)

# Add occasional slow requests (GC pauses, cold cache)
n_slow = int(0.01 * n_requests)
slow_indices = np.random.choice(n_requests, n_slow, replace=False)
latencies[slow_indices] *= np.random.uniform(3, 10, n_slow)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Latency histogram with percentile lines
axes[0, 0].hist(latencies, bins=100, density=True, alpha=0.7, color='#3498db', edgecolor='white')
for p, color, label in [(50, 'green', 'p50'), (95, 'orange', 'p95'), (99, 'red', 'p99')]:
    val = np.percentile(latencies, p)
    axes[0, 0].axvline(x=val, color=color, linestyle='--', linewidth=2, label=f'{label}={val:.0f}ms')
axes[0, 0].set_xlabel('Latency (ms)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Inference Latency Distribution')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 200)
axes[0, 0].grid(True, alpha=0.3)

# SLO burn-down over time
slo_target = 100  # ms
window_size = 100
slo_compliance = np.array([np.mean(latencies[max(0,i-window_size):i] <= slo_target) 
                           for i in range(1, n_requests+1)])

axes[0, 1].plot(slo_compliance * 100, 'b-', linewidth=0.5, alpha=0.7)
axes[0, 1].axhline(y=99, color='red', linestyle='--', label='SLO Target (99%)')
axes[0, 1].set_xlabel('Request Number')
axes[0, 1].set_ylabel('SLO Compliance (%)')
axes[0, 1].set_title('Rolling SLO Compliance (window=100)')
axes[0, 1].set_ylim(90, 100.5)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Error budget consumption
error_budget_total = 43.2  # minutes per month
days = np.arange(1, 31)
daily_violations = np.random.poisson(0.5, 30)  # minutes of SLO violation per day
cumulative_consumed = np.cumsum(daily_violations)
budget_remaining = error_budget_total - cumulative_consumed

axes[1, 0].bar(days, daily_violations, color='#e74c3c', alpha=0.7, label='Daily violations')
ax2 = axes[1, 0].twinx()
ax2.plot(days, budget_remaining, 'b-o', markersize=4, label='Budget remaining')
ax2.axhline(y=0, color='red', linestyle=':', alpha=0.5)
axes[1, 0].set_xlabel('Day of Month')
axes[1, 0].set_ylabel('Violations (minutes)', color='red')
ax2.set_ylabel('Budget Remaining (minutes)', color='blue')
axes[1, 0].set_title('Error Budget Consumption')
axes[1, 0].legend(loc='upper left')
ax2.legend(loc='upper right')

# Throughput over time
time_minutes = np.arange(0, 60)
qps = 100 + 30 * np.sin(2 * np.pi * time_minutes / 60) + np.random.randn(60) * 5
axes[1, 1].plot(time_minutes, qps, 'g-', linewidth=1.5)
axes[1, 1].fill_between(time_minutes, qps, alpha=0.2, color='green')
axes[1, 1].axhline(y=150, color='red', linestyle='--', label='Capacity limit')
axes[1, 1].set_xlabel('Time (minutes)')
axes[1, 1].set_ylabel('Throughput (QPS)')
axes[1, 1].set_title('Real-time Throughput Monitoring')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cloud_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Latency p50={np.percentile(latencies, 50):.1f}ms, p99={np.percentile(latencies, 99):.1f}ms")
print(f"SLO compliance: {np.mean(latencies <= slo_target)*100:.2f}%")
print(f"Error budget remaining: {budget_remaining[-1]:.1f} minutes")

<a id='9'></a>
## 9. Cost Optimization

### Cost Model for Cloud ML Serving

$$\text{Cost}_{\text{monthly}} = N_{\text{replicas}} \cdot C_{\text{instance}} \cdot T_{\text{hours}} + C_{\text{network}} + C_{\text{storage}}$$

The cost-per-inference:

$$C_{\text{per\_request}} = \frac{\text{Cost}_{\text{monthly}}}{\lambda \cdot T_{\text{seconds/month}}} = \frac{N \cdot C_{\text{hr}}}{\lambda \cdot 3600}$$

### Instance Type Selection

| Instance | GPU | Cost/hr | Peak QPS | Cost/1M requests |
|----------|-----|---------|----------|------------------|
| c5.4xlarge (CPU) | — | $0.68 | 50 | $3.78 |
| g4dn.xlarge (T4) | T4 16GB | $0.526 | 200 | $0.73 |
| g5.xlarge (A10G) | A10G 24GB | $1.006 | 500 | $0.56 |
| p4d.24xlarge (A100) | 8×A100 | $32.77 | 5000 | $1.82 |

### Spot Instance Strategy

Spot instances offer 60-90% savings but can be interrupted. For ML serving:

$$\text{Effective Cost} = C_{\text{spot}} \cdot (1 + P_{\text{interrupt}} \cdot \text{penalty}_{\text{recovery}})$$

**Hybrid approach:** Maintain on-demand baseline + spot overflow:

$$N_{\text{on-demand}} = \lceil \lambda_{\text{min}} / \text{QPS}_{\text{per\_replica}} \rceil$$
$$N_{\text{spot}} = \lceil (\lambda_{\text{current}} - \lambda_{\text{min}}) / \text{QPS}_{\text{per\_replica}} \rceil$$

### Precision vs Cost Tradeoff

Lower precision = higher throughput = fewer instances:

| Precision | Relative Throughput | Cost Reduction |
|-----------|--------------------|-----------------|
| FP32 | 1.0× | Baseline |
| FP16 | 1.8–2.0× | 45–50% |
| INT8 | 2.5–4.0× | 60–75% |

$$\text{Savings} = 1 - \frac{1}{\text{Speedup}} = 1 - \frac{N_{\text{FP32}}}{N_{\text{quantized}}}$$

### Right-Sizing Algorithm

Find the cheapest instance that meets SLO:

$$\text{optimal} = \arg\min_{i \in \text{instances}} \; C_i \cdot N_i \quad \text{s.t.} \quad N_i \cdot \text{QPS}_i \geq \lambda_{\text{target}} \; \land \; T_{p99,i} \leq T_{\text{SLA}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cost optimization analysis
instances = {
    'c5.4xl (CPU)': {'cost_hr': 0.68, 'qps': 50, 'color': '#95a5a6'},
    'g4dn.xl (T4)': {'cost_hr': 0.526, 'qps': 200, 'color': '#2ecc71'},
    'g5.xl (A10G)': {'cost_hr': 1.006, 'qps': 500, 'color': '#3498db'},
    'p4d.24xl (8×A100)': {'cost_hr': 32.77, 'qps': 5000, 'color': '#9b59b6'},
}

target_qps_range = np.linspace(10, 2000, 100)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Cost per 1M requests vs target QPS
for name, spec in instances.items():
    n_replicas = np.ceil(target_qps_range / spec['qps'])
    monthly_cost = n_replicas * spec['cost_hr'] * 730  # hours/month
    cost_per_million = monthly_cost / (target_qps_range * 3600 * 730 / 1e6)
    axes[0].plot(target_qps_range, cost_per_million, linewidth=2, 
                color=spec['color'], label=name)

axes[0].set_xlabel('Target QPS')
axes[0].set_ylabel('Cost per 1M Requests ($)')
axes[0].set_title('Cost Efficiency by Instance Type')
axes[0].legend(fontsize=9)
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.1, 100)

# Spot vs On-Demand cost comparison
hours = np.arange(0, 730)  # One month
on_demand_rate = 1.006  # g5.xlarge
spot_rate = 0.35  # ~65% discount
interrupt_rate = 0.05  # 5% chance per hour
recovery_cost = 2.0  # Cost of interrupted request + cold start

on_demand_cost = np.cumsum(np.ones(730) * on_demand_rate)
spot_cost = np.cumsum(np.ones(730) * spot_rate)
# Add recovery costs from interruptions
interrupts = np.random.binomial(1, interrupt_rate, 730)
spot_with_recovery = np.cumsum(spot_rate + interrupts * recovery_cost)

axes[1].plot(hours, on_demand_cost, 'r-', linewidth=2, label='On-Demand')
axes[1].plot(hours, spot_cost, 'g--', linewidth=2, label='Spot (no interrupts)')
axes[1].plot(hours, spot_with_recovery, 'g-', linewidth=2, label='Spot (with recovery)')
axes[1].set_xlabel('Hours')
axes[1].set_ylabel('Cumulative Cost ($)')
axes[1].set_title('On-Demand vs Spot Instance Cost (g5.xl)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Precision impact on cost
precisions = ['FP32', 'FP16', 'INT8', 'INT4']
throughput_multipliers = [1.0, 1.9, 3.2, 5.0]
accuracy_penalties = [0, 0.1, 0.5, 2.0]  # % accuracy drop

base_cost = 1000  # $/month for FP32
costs = [base_cost / m for m in throughput_multipliers]

ax1 = axes[2]
ax2_twin = ax1.twinx()

bars = ax1.bar(precisions, costs, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'], alpha=0.7)
ax2_twin.plot(precisions, accuracy_penalties, 'ko-', linewidth=2, markersize=8)

ax1.set_ylabel('Monthly Cost ($)', color='blue')
ax2_twin.set_ylabel('Accuracy Drop (%)', color='black')
ax1.set_title('Cost vs Accuracy: Precision Tradeoff')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cost_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print("Cost per 1M requests at 500 QPS:")
for name, spec in instances.items():
    n = int(np.ceil(500 / spec['qps']))
    cost_1m = n * spec['cost_hr'] / (500 * 3.6)
    print(f"  {name}: {n} replicas, ${cost_1m:.2f}/1M requests")

<a id='10'></a>
## 10. Production Deployment Patterns

### Blue-Green Deployment

```
┌─────────────────────────────────────────────────────────────┐
│                     Load Balancer                             │
│              ┌──────────┐                                    │
│              │  Router  │                                    │
│              └─────┬────┘                                    │
│                    │                                          │
│         ┌──────────┼──────────┐                              │
│         ▼          │          ▼                              │
│  ┌─────────────┐   │   ┌─────────────┐                      │
│  │  BLUE (v1)  │   │   │ GREEN (v2)  │                      │
│  │  (active)   │◀──┘   │ (standby)   │                      │
│  │ 5 replicas  │       │ 5 replicas  │                      │
│  └─────────────┘       └─────────────┘                      │
│                                                              │
│  After validation: swap router to GREEN                      │
└─────────────────────────────────────────────────────────────┘
```

### Canary Deployment with Statistical Validation

Gradually shift traffic while monitoring for regressions:

$$\text{canary\_weight}(t) = \min\left(w_{\max}, \; w_0 \cdot 2^{t/T_{\text{double}}}\right)$$

**Statistical test for promotion:** Compare canary vs baseline using a two-proportion z-test:

$$z = \frac{\hat{p}_{\text{canary}} - \hat{p}_{\text{baseline}}}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_1} + \frac{1}{n_2}\right)}}$$

Promote if $|z| < z_{\alpha/2}$ (no significant difference) with sufficient sample size:

$$n \geq \frac{(z_{\alpha/2} + z_\beta)^2 \cdot 2\hat{p}(1-\hat{p})}{\delta^2}$$

### Shadow Deployment (Dark Launch)

```
  Request ──┬──▶ Production Model v1 ──▶ Response to client
            │
            └──▶ Shadow Model v2 ──▶ Log (no client response)
                                       Compare offline
```

Shadow deployment is ideal for ML models because:
- No risk to production quality
- Captures real traffic distribution (unlike synthetic tests)
- Can detect drift between model versions on production data

### Model Versioning Strategy

```
  Model Registry:
    model-v1.0.onnx  [production]  deployed: 2024-01-15
    model-v1.1.onnx  [canary]      deployed: 2024-02-01  (5% traffic)
    model-v2.0.onnx  [shadow]      deployed: 2024-02-10  (logging only)
    model-v2.1.onnx  [staged]      pending validation
```

### Rollback Criteria

Automatic rollback triggers:

$$\text{rollback if } \begin{cases}
T_{p99} > 2 \cdot T_{p99}^{\text{baseline}} & \text{(latency regression)} \\
\text{error\_rate} > \text{error\_rate}^{\text{baseline}} + 3\sigma & \text{(quality regression)} \\
\text{OOM events} > 0 & \text{(stability failure)}
\end{cases}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Canary deployment simulation
np.random.seed(42)

# Simulate canary rollout over time
hours = np.arange(0, 48)  # 48-hour canary
doubling_time = 6  # hours
w_initial = 0.01  # Start at 1%
w_max = 1.0

canary_weight = np.minimum(w_max, w_initial * 2**(hours / doubling_time))

# Simulate metrics for baseline and canary
n_requests_per_hour = 10000
baseline_error_rate = 0.005  # 0.5% baseline error
canary_error_rate_good = 0.004  # Good canary (slightly better)
canary_error_rate_bad = 0.015  # Bad canary (3x worse)

# Scenario 1: Good canary
baseline_errors = np.random.binomial(n_requests_per_hour, baseline_error_rate, len(hours))
canary_good_errors = np.random.binomial(
    (n_requests_per_hour * canary_weight).astype(int), 
    canary_error_rate_good, len(hours))

# Scenario 2: Bad canary (simulating regression)
canary_bad_errors = np.random.binomial(
    (n_requests_per_hour * canary_weight).astype(int),
    canary_error_rate_bad, len(hours))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Canary weight over time
axes[0, 0].plot(hours, canary_weight * 100, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Hours since deployment')
axes[0, 0].set_ylabel('Canary Traffic (%)')
axes[0, 0].set_title('Canary Traffic Ramp-up (doubling every 6h)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_yscale('log')
axes[0, 0].set_ylim(0.5, 150)
axes[0, 0].axhline(y=100, color='green', linestyle='--', alpha=0.5, label='Full rollout')
axes[0, 0].legend()

# Good canary: error rates converge
canary_n = (n_requests_per_hour * canary_weight).astype(int)
canary_good_rate = np.where(canary_n > 0, canary_good_errors / canary_n, 0) * 100
baseline_rate = baseline_errors / n_requests_per_hour * 100

axes[0, 1].plot(hours, baseline_rate, 'b-', linewidth=1.5, alpha=0.7, label='Baseline (v1)')
axes[0, 1].plot(hours, canary_good_rate, 'g-', linewidth=1.5, alpha=0.7, label='Canary (v2) - Good')
axes[0, 1].set_xlabel('Hours')
axes[0, 1].set_ylabel('Error Rate (%)')
axes[0, 1].set_title('Scenario: Successful Canary')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Bad canary: error rate diverges
canary_bad_rate = np.where(canary_n > 0, canary_bad_errors / canary_n, 0) * 100
axes[1, 0].plot(hours, baseline_rate, 'b-', linewidth=1.5, alpha=0.7, label='Baseline (v1)')
axes[1, 0].plot(hours, canary_bad_rate, 'r-', linewidth=1.5, alpha=0.7, label='Canary (v2) - Bad')
# Mark rollback point
rollback_hour = next((h for h in range(len(hours)) if canary_bad_rate[h] > baseline_rate[h] + 0.5 and canary_n[h] > 100), None)
if rollback_hour:
    axes[1, 0].axvline(x=rollback_hour, color='red', linestyle='--', linewidth=2)
    axes[1, 0].annotate(f'Auto-rollback\nh={rollback_hour}', 
                        xy=(rollback_hour, canary_bad_rate[rollback_hour]),
                        xytext=(rollback_hour+5, canary_bad_rate[rollback_hour]+0.3),
                        arrowprops=dict(arrowstyle='->', color='red'))
axes[1, 0].set_xlabel('Hours')
axes[1, 0].set_ylabel('Error Rate (%)')
axes[1, 0].set_title('Scenario: Failed Canary (Auto-rollback)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Statistical power: required sample size vs effect size
effect_sizes = np.linspace(0.001, 0.02, 50)
alpha = 0.05
beta = 0.2
z_alpha = 1.96
z_beta = 0.84
p_baseline = 0.005

sample_sizes = (z_alpha + z_beta)**2 * 2 * p_baseline * (1 - p_baseline) / effect_sizes**2

axes[1, 1].plot(effect_sizes * 100, sample_sizes, 'b-', linewidth=2)
axes[1, 1].set_xlabel('Minimum Detectable Effect (%)')
axes[1, 1].set_ylabel('Required Sample Size (per group)')
axes[1, 1].set_title('Statistical Power: Sample Size Requirements\n(α=0.05, β=0.2, p₀=0.5%)')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=10000, color='orange', linestyle='--', label='10K requests')
axes[1, 1].axhline(y=100000, color='red', linestyle='--', label='100K requests')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('deployment_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"To detect 0.5% effect size change: need {sample_sizes[24]:.0f} samples per group")
print(f"At 100 QPS with 10% canary: takes {sample_sizes[24]/(100*0.1*3600):.1f} hours")

## 11. Production FastAPI Serving Example

The following demonstrates a production-grade serving pattern with:
- Session pre-loading with warmup
- Health and readiness endpoints
- Request validation and error handling
- Latency tracking with percentile computation
- Batch inference support

In [ ]:
import numpy as np
import time
import onnxruntime as ort
from collections import deque

class CloudInferenceServer:
    """Production-grade ONNX inference server with monitoring."""
    
    def __init__(self, model_path=None, max_batch_size=32):
        self.max_batch_size = max_batch_size
        self.latency_window = deque(maxlen=10000)
        self.request_count = 0
        self.error_count = 0
        self.session = None
        self.ready = False
        
        if model_path:
            self._load_model(model_path)
    
    def _load_model(self, path):
        """Load model with cloud-optimized settings."""
        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        so.execution_mode = ort.ExecutionMode.ORT_PARALLEL
        so.enable_mem_pattern = True
        
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        self.session = ort.InferenceSession(path, so, providers=providers)
        self._warmup()
        self.ready = True
    
    def _warmup(self, n_warmup=5):
        """Warmup inference to trigger JIT compilation and memory allocation."""
        input_meta = self.session.get_inputs()[0]
        shape = [1 if isinstance(d, str) else d for d in input_meta.shape]
        dummy = np.random.randn(*shape).astype(np.float32)
        for _ in range(n_warmup):
            self.session.run(None, {input_meta.name: dummy})
    
    def infer(self, inputs: dict) -> dict:
        """Run inference with latency tracking."""
        start = time.perf_counter()
        try:
            outputs = self.session.run(None, inputs)
            elapsed_ms = (time.perf_counter() - start) * 1000
            self.latency_window.append(elapsed_ms)
            self.request_count += 1
            return {'outputs': outputs, 'latency_ms': elapsed_ms}
        except Exception as e:
            self.error_count += 1
            raise
    
    def get_metrics(self) -> dict:
        """Return current serving metrics."""
        latencies = np.array(self.latency_window) if self.latency_window else np.array([0])
        return {
            'total_requests': self.request_count,
            'total_errors': self.error_count,
            'error_rate': self.error_count / max(self.request_count, 1),
            'latency_p50_ms': float(np.percentile(latencies, 50)),
            'latency_p95_ms': float(np.percentile(latencies, 95)),
            'latency_p99_ms': float(np.percentile(latencies, 99)),
            'ready': self.ready,
        }

# Demo (without actual model file)
server = CloudInferenceServer()
print("CloudInferenceServer initialized (no model loaded - demo mode)")
print(f"Max batch size: {server.max_batch_size}")
print(f"Ready: {server.ready}")

# Simulate metrics from a running server
server.request_count = 50000
server.error_count = 23
server.latency_window = deque(np.random.lognormal(np.log(25), 0.4, 10000))
metrics = server.get_metrics()
print(f"\nSimulated metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Summary

Cloud deployment of ONNX models is a systems engineering challenge that extends far beyond model inference. The key mathematical frameworks covered:

### Core Equations

| Concept | Formula |
|---------|--------|
| **Throughput** | $\text{QPS} = \frac{B \cdot N_{\text{replicas}}}{T_{p99}}$ |
| **Little's Law** | $L = \lambda \cdot W$ |
| **Scaling** | $N = \lceil N_{\text{curr}} \cdot U_{\text{curr}} / U_{\text{target}} \rceil$ |
| **Batch wait** | $\mathbb{E}[T_{\text{wait}}] = \tau/2$ (under timeout) |
| **Cost/request** | $C = N \cdot C_{\text{hr}} / (\lambda \cdot 3600)$ |

### Key Takeaways

1. **Batching is the primary GPU efficiency lever** — dynamic batching with proper timeout tuning can 10× throughput
2. **Auto-scaling must be GPU-aware** — CPU utilization is misleading for GPU workloads
3. **Monitoring drives reliability** — RED metrics, error budgets, and drift detection are non-negotiable
4. **Cost optimization requires precision awareness** — FP16/INT8 can cut costs 50-75% with minimal accuracy loss
5. **Deployment patterns must be statistical** — canary validation requires proper sample size calculation

---

*Next: [Mobile Deployment](../03_Mobile_Deployment/Mobile_Deployment_Deep_Dive.ipynb) explores constrained inference on Android/iOS with NNAPI and CoreML execution providers.*